In [ ]:
import pathlib as pl

import jupyter_black
import pandas as pd
import xarray as xr

import pywatershed as pws

jupyter_black.load()
pd.set_option("display.max_rows", 500)

In [ ]:
# inputs
data_dir = pl.Path("../../data")

starfit_data_dir = data_dir / "starfit"
resnat_data_dir = starfit_data_dir / "nknowles_resnat/data/updated_locally"
starfit_dir = resnat_data_dir / "ISTARF"
grand_dir = resnat_data_dir / "GRanD_Version_1_3"
res_data_dir = starfit_data_dir / "ResOpsUS"
grand_crosswalk_dir = starfit_data_dir / "grand/Crosswalktableb"

domain_dir = data_dir / "domain_files/20250305_v1.1_gm_byHWobs_republican"
gis_dir = domain_dir / "GIS_v3"

# output
starfit_param_file_name = pl.Path(
    "../../data/domain_files/pws_preprocessed/"
    "starfit_params_republican_one_res_above_seg_NEW.nc"
)


# input files
control_file = domain_dir / "control.default.bandit"
grand_nhm_seg_crosswalk_file = grand_crosswalk_dir / "istarf_xwalk.csv"
grand_file = grand_dir / "GRanD_dams_v1_3.shp"
prms_gpkg_file = gis_dir / "model_layers.gpkg"  # "model_nsegment.shp"
starfit_param_csv_file = starfit_dir / "ISTARF-CONUS.csv"
resops_inflow_file = (
    res_data_dir
    / "ResOpsUS/time_series_single_variable_table/DAILY_AV_INFLOW_CUMECS.csv"
)
resops_outflow_file = (
    res_data_dir
    / "ResOpsUS/time_series_single_variable_table/DAILY_AV_OUTFLOW_CUMECS.csv"
)
resops_storage_file = (
    res_data_dir / "ResOpsUS/time_series_single_variable_table/DAILY_AV_STORAGE_MCM.csv"
)

In [ ]:
assert control_file.exists()
assert grand_nhm_seg_crosswalk_file.exists()
assert grand_file.exists()
assert prms_gpkg_file.exists()
assert starfit_param_csv_file.exists()

In [ ]:
mk_sf_params = pws.MakeStarfitParams(
    control_file=control_file,
    grand_nhm_seg_crosswalk_file=grand_nhm_seg_crosswalk_file,
    grand_file=grand_file,
    starfit_param_csv_file=starfit_param_csv_file,
    resops_inflow_file=resops_inflow_file,
    resops_outflow_file=resops_outflow_file,
    resops_storage_file=resops_storage_file,
    prms_gpkg_file=prms_gpkg_file,  # optional, keep at bottom
)

In [ ]:
mk_sf_params.plot_grand_in_domain()

In [ ]:
mk_sf_params.drop_grand_ids([333, 440])

In [ ]:
mk_sf_params.rm_small_duplicates()
mk_sf_params.plot_grand_in_domain()

In [ ]:
mk_sf_params.dataset

In [ ]:
mk_sf_params.to_netcdf(starfit_param_file_name)

In [ ]:
check_file = pl.Path(
    "../../data/domain_files/pws_preprocessed/"
    "starfit_params_republican_one_res_above_seg.nc"
)

In [ ]:
result = xr.load_dataset(starfit_param_file_name)
answer = xr.load_dataset(check_file)

In [ ]:
xr.testing.assert_equal(result, answer)